# 04 · Optimize inference cost and latency

We will remove unnecessary inference round trips, bound the answer length,
and reuse a stable prompt prefix with caching. Each change gets a measured
before/after run; inspect answer quality before accepting the savings.

In [ ]:
import json
from pathlib import Path
from IPython.display import display
from utils.workshop import Rates, Run, Settings, compare

settings = Settings.from_env()
rates = Rates.load(model_id=settings.model_id)
order = json.loads(Path("data/order.json").read_text())
facts = {key: order[key] for key in ["order_id", "status", "days_late"]}
policy = Path("data/knowledge-base/delivery-policy.txt").read_text()
evidence = f"Order: {json.dumps(facts)}\nPolicy [S1]: {policy}"

## Edit the prompts

The staged and fused versions receive the same `system_prompt`, `instruction`, and evidence. Stage-specific prompts and the concise variant are also editable; no function default freezes an earlier prompt value.

Rerun this cell after edits, then rerun the relevant example. For comparisons, rerun both baseline and tuned versions.

In [ ]:
system_prompt = (
    "Use the supplied order and policy to explain replacement eligibility "
    "and the next support action. Include the order ID, delay, and policy conditions."
)
instruction = "Can this order be replaced, and what should support do next?"
stage_prompts = {
    "classify": "Classify the customer's issue in a short label.",
    "assess": "Assess replacement eligibility using the supplied policy.",
}

## 1. Baseline: three calls using evidence we already have

All facts and policy text are available before generation. This makes the
extra classification and assessment round trips candidates for consolidation.

In [ ]:
def run_staged(label, system_prompt, instruction, evidence, stage_prompts):
    task = f"{evidence}\nQuestion: {instruction}"
    with Run(label, settings, rates) as run:
        issue = run.generate(
            "classify", task, instructions=stage_prompts["classify"],
        )
        assessment = run.generate(
            "assess", f"{task}\nIssue: {issue.text}",
            instructions=stage_prompts["assess"],
        )
        answer = run.generate(
            "draft", f"{task}\nAssessment: {assessment.text}",
            instructions=system_prompt, max_output_tokens=4096,
        )
    return run, answer.text

baseline_run, baseline_answer = run_staged(
    "04_staged",
    system_prompt=system_prompt, instruction=instruction,
    evidence=evidence, stage_prompts=stage_prompts,
)
print(baseline_answer)
display(baseline_run.metrics())
baseline_run.save()

## 2. Consolidate the work into one call

Keep the same evidence and final-answer instructions. We can consolidate
these stages because there is no external tool result to wait for between them.

In [ ]:
def run_fused(label, system_prompt, instruction, evidence, output_budget=4096):
    task = f"{evidence}\nQuestion: {instruction}"
    with Run(label, settings, rates) as run:
        answer = run.generate(
            "answer", task, instructions=system_prompt,
            max_output_tokens=output_budget,
        )
    return run, answer.text

fused_run, fused_answer = run_fused(
    "04_fused",
    system_prompt=system_prompt, instruction=instruction, evidence=evidence,
)
print(fused_answer)
display(compare(baseline_run, fused_run))
fused_run.save()

## 3. Ask for a shorter useful answer

Add a two-sentence instruction and lower the output ceiling. The instruction
drives brevity; the ceiling alone does not guarantee fewer tokens and can
produce an incomplete response if set too low.

In [ ]:
concise_system_prompt = (
    system_prompt
    + " Answer in two sentences without losing the conditions."
)

In [ ]:
concise_run, concise_answer = run_fused(
    "04_concise",
    system_prompt=concise_system_prompt, instruction=instruction,
    evidence=evidence, output_budget=2048,
)
print(concise_answer)
display(compare(baseline_run, fused_run, concise_run))
concise_run.save()

**Quality check:** all answers should retain the order ID, four-day delay,
in-transit status, and the policy's three-day condition. A faster answer that
loses these facts is not a successful optimization.

## 4. Make the prompt layout reusable

A request without our explicit checkpoint can still report cached tokens,
as the rehearsal showed. We now keep an explicit checkpoint in both variants
and change **prompt layout**: request-specific metadata before the reference
versus after it.

The first layout is a deliberately cache-unfriendly baseline, not the service's
default behavior. We include the same request ID, question, and reference in
each variant; if the model does not need request metadata, omit it entirely.

In [ ]:
from uuid import uuid4
from pandas import DataFrame

reference = Path("data/cache_reference.md").read_text()
cache_system_prompt = "Answer from the supplied evidence in two sentences."
cache_questions = [
    f"{evidence}\nCan this order be replaced?",
    f"{evidence}\nExplain the policy conditions that justify the decision.",
]
repetitions = 2
questions = cache_questions * repetitions
batch_id = uuid4().hex
request_metadata = [
    f"Request-ID: {batch_id}-{number}"
    for number in range(1, len(questions) + 1)
]
cache_output_budget = 2048
RUN_CHECKPOINT_COMPARISON = True
print("Requests per variant:", len(questions))

Edit the prompt settings above, then keep them fixed across the variants.
The metadata is generated once and reused in every variant; this is an
openly controlled layout experiment, not a claim that real traffic always misses.

In [ ]:
def run_cache_workload(
    label, system_prompt, questions, reference, request_metadata,
    layout, mode, output_budget,
):
    """Stream the same workload with the selected prefix layout and checkpoint."""
    if len(questions) < 2:
        raise ValueError("Use at least two questions to observe prefix reuse.")
    if len(questions) != len(request_metadata):
        raise ValueError("Provide one metadata string per question.")
    if layout not in {"dynamic_first", "stable_first"}:
        raise ValueError("Unknown prompt layout.")
    answers = []
    with Run(label, settings, rates) as run:
        for number, (question, metadata) in enumerate(
            zip(questions, request_metadata), 1
        ):
            if layout == "dynamic_first":
                prefix = f"{metadata}\n\n{reference}"
                prompt = question
            else:
                prefix = reference
                prompt = f"{metadata}\n\n{question}"
            response = run.stream(
                f"question_{number}", prompt, instructions=system_prompt,
                cache_prefix=prefix, cache_mode=mode,
                max_output_tokens=output_budget,
            )
            answers.append(response.text)
    return run, answers

Both variants stream internally so we can measure **time to first visible text**,
then wait for final token usage. Total duration is also reported, but it
includes output generation and is not a pure measurement of prefix processing.

In [ ]:
dynamic_run, dynamic_answers = run_cache_workload(
    "04_dynamic_prefix",
    system_prompt=cache_system_prompt, questions=questions,
    reference=reference, request_metadata=request_metadata,
    layout="dynamic_first", mode="on", output_budget=cache_output_budget,
)
stable_run, stable_answers = run_cache_workload(
    "04_stable_prefix",
    system_prompt=cache_system_prompt, questions=questions,
    reference=reference, request_metadata=request_metadata,
    layout="stable_first", mode="on", output_budget=cache_output_budget,
)
print(stable_answers[-1])

**Look for:** repeated writes when metadata changes before the checkpoint,
and later reads when the prefix stays stable. Both variants request the same
checkpoint policy; use returned counters to check what actually happened.

In [ ]:
def cache_measurements(run):
    """Separate input/cache charges from variable output-generation charges."""
    frame = run.frame().copy()
    frame["input_usd"] = float("nan")
    if run.rates is not None:
        output_cost = (
            frame["output_tokens"] * run.rates.output_per_million / 1_000_000
        )
        frame["input_usd"] = frame["reported_usage_cost_usd"] - output_cost
    return frame

cache_columns = [
    "step", "cached_input_tokens", "cache_write_tokens", "output_tokens",
    "time_to_first_text_s", "elapsed_s", "input_usd",
]
print("Request metadata before the reference")
display(cache_measurements(dynamic_run)[cache_columns])
print("Stable reference before request metadata")
display(cache_measurements(stable_run)[cache_columns])
display(compare(dynamic_run, stable_run))
dynamic_run.save()
stable_run.save()

`input_usd` isolates input/cache charges by subtracting the reported output
cost, so answer-length differences do not masquerade as cache savings.
`compare()` still shows the full observed cost of each request sequence.

## 5. Optional: does this model need an explicit checkpoint?

Keep the stable layout and remove the developer checkpoint for a third run.
Set `RUN_CHECKPOINT_COMPARISON = False` to skip it during the talk; caching
already provided by the model may make the checkpoint less useful.

In [ ]:
cache_runs = {"Dynamic + checkpoint": dynamic_run, "Stable + checkpoint": stable_run}
if RUN_CHECKPOINT_COMPARISON:
    unmarked_run, unmarked_answers = run_cache_workload(
        "04_stable_without_checkpoint",
        system_prompt=cache_system_prompt, questions=questions,
        reference=reference, request_metadata=request_metadata,
        layout="stable_first", mode="off",
        output_budget=cache_output_budget,
    )
    cache_runs["Stable, no checkpoint"] = unmarked_run
    print(unmarked_answers[-1])
    display(cache_measurements(unmarked_run)[cache_columns])
    display(compare(stable_run, unmarked_run))
    unmarked_run.save()

`mode="off"` omits our checkpoint; it does not prove a cache miss. Compare
first-request cost with later requests and inspect reads/writes: the earlier
run or a previous rehearsal can leave the reference warm.

In [ ]:
rows = []
for label, run in cache_runs.items():
    frame = cache_measurements(run)
    later = frame.iloc[1:]
    rows.append({
        "Variant": label,
        "First input USD": frame.iloc[0]["input_usd"],
        "Later median input USD": later["input_usd"].median(),
        "Later median first text (s)": later["time_to_first_text_s"].median(),
        "Later median duration (s)": later["elapsed_s"].median(),
        "Reported sequence USD": run.summary()["reported_usage_cost_usd"],
    })
display(DataFrame(rows).set_index("Variant"))

### Optional: estimate the checkpoint's input-cost break-even

The projection below uses paired later-request input costs. It applies only
when both later sequences actually hit cache, with one observed checkpoint
write and no later writes; it is not a measured latency or billing guarantee.

In [ ]:
from math import ceil

if RUN_CHECKPOINT_COMPARISON:
    base = cache_measurements(unmarked_run)
    marked = cache_measurements(stable_run)
    costs_known = base["input_usd"].notna().all() and marked["input_usd"].notna().all()
    later_hits = (
        (base.iloc[1:]["cached_input_tokens"] > 0).all()
        and (marked.iloc[1:]["cached_input_tokens"] > 0).all()
    )
    one_write = (
        marked.iloc[0]["cache_write_tokens"] > 0
        and marked.iloc[1:]["cache_write_tokens"].sum() == 0
        and base.iloc[1:]["cache_write_tokens"].sum() == 0
    )
    if costs_known and later_hits and one_write:
        initial_extra = marked.iloc[0]["input_usd"] - base.iloc[0]["input_usd"]
        reuse_saving = (base.iloc[1:]["input_usd"] - marked.iloc[1:]["input_usd"]).median()
        if initial_extra > 0 and reuse_saving > 0:
            print("Projected total requests to break even:", 1 + ceil(initial_extra / reuse_saving))
        elif reuse_saving <= 0:
            print("No positive per-reuse input saving was observed for the checkpoint.")
        else:
            print("No initial premium was observed; inspect the cache's starting state.")
    else:
        print("Insufficient comparable cache/cost observations for a break-even projection.")

The cached content must still be useful; do not add a large prefix merely
to obtain a cache hit. Review the answers, repeat or reverse run order during
rehearsal, and report the result even when a policy is slower or more expensive.

**Takeaway:** the staged-to-fused and concise-answer experiments remove work.
The cache experiment changes how needed, repeated context is processed.
Source: [AWS prompt caching](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html).